<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/11-representation-self-supervised-learning.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **表示学习与自监督学习** {#representation-self-supervised-learning}

前面的章节关注哪种架构适合空间、序列、注意力或关系结构。本章改变问题：**当标签稀缺、昂贵、覆盖范围狭窄或完全不可用时，架构应该在 embedding 中保留哪些信息？**表示学习用学习到的映射 $f_\theta:x\mapsto h$ 取代人工规定的特征。自监督学习（self-supervised learning，SSL）从数据中已经存在的关系构造训练目标，例如同一图像的两种 augmentation、被遮挡区域及其上下文、相邻模态，或缓慢变化的 teacher。

区分这些概念很重要。表示学习是广义目标；自监督只是学习信号的一种来源。Triplet learning 可以使用人工类别标签，CLIP 使用配对 image-text supervision，masked autoencoding 不使用类别标签，而 knowledge distillation 传递 teacher output。它们都能产生 embedding，但各自的 invariance 和 failure mode 来自不同的监督来源。

本章使用 `load_digits` 中的 [UCI Optical Recognition of Handwritten Digits dataset](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B) 副本。官方 [scikit-learn 文档](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)说明它包含 1,797 张 8×8 图像，像素为 0 到 16 的整数。UCI 将数据归属 E. Alpaydin 与 C. Kaynak，DOI 为 [10.24432/C50P49](https://doi.org/10.24432/C50P49)，许可证为 CC BY 4.0。整章复用固定的分层 70/15/15 划分。SSL 预训练只看到训练图像；类别标签只进入明确标注为监督式的 metric learning、prompt alignment、probe 和 evaluation。

![共享 UCI 衍生数据集中每个类别的一张 8×8 手写数字图像。](assets/dl11-digits-samples.svg){fig-align="center" width="76%" fig-alt="十张灰度 optical digit 小图，分别表示从零到九的类别。"}

*根据 scikit-learn 的 [UCI 衍生 digits 副本](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html)生成的原创数据可视化；UCI 数据 DOI 10.24432/C50P49，CC BY 4.0。*

### **什么构成有用的表示** {#what-makes-useful-representation}

有用表示应保留多个下游决策需要的因素，同时抑制无关变化。对数字身份而言，轻微传感器噪声和缺失像素可以是 nuisance，笔画拓扑通常具有信息量。这产生两个相互竞争的要求：**不变性（invariance）**，即允许的变换不会显著改变 $h$；以及**选择性（selectivity）**，即语义不同的输入仍可区分。常数向量具有完美不变性，却毫无用途。Raw pixels 选择性很强，却容易受无害扰动影响。

表示的效用取决于任务族与部署分布。为 digit class 优化的 embedding 可能故意丢弃书写风格，这对 writer identification 反而有害。不存在普遍充分的表示。因此，良好实践应声明目标 invariance，使用多个 downstream task 或 probe 评估，在真实 shift 下检查 robustness，并测量信息是否坍缩到过少维度。

![PCA 表明即使简单表示也会定义邻域与类别重叠。](assets/dl11-digits-pca.svg){fig-align="center" width="76%" fig-alt="缩放手写数字像素的二维 PCA 散点图，颜色仅用于解释数字类别。"}

*800 张缩放 digit image 的原创可视化。PCA 拟合没有使用标签；颜色只用于检查形成的几何结构。*

<details>
<summary><strong>PyTorch：建立共享 digits split 与 augmentation contract</strong></summary>

```python
import copy
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1110):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).unsqueeze(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
all_indices = np.arange(len(all_images))

train_idx, holdout_idx = train_test_split(
    all_indices, test_size=0.30, random_state=1110, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1110,
    stratify=digits.target[holdout_idx],
)
train_images, train_labels = all_images[train_idx], all_labels[train_idx]
val_images, val_labels = all_images[val_idx], all_labels[val_idx]
test_images, test_labels = all_images[test_idx], all_labels[test_idx]


def augment_batch(images, noise_std=0.08, drop_probability=0.08):
    # Independent contrast/noise/dropout views preserve the coarse 8x8 digit shape.
    contrast = torch.empty(len(images), 1, 1, 1).uniform_(0.85, 1.15)
    noisy = images * contrast + noise_std * torch.randn_like(images)
    keep = torch.rand_like(images) > drop_probability
    return (noisy * keep).clamp(0.0, 1.0)


class DigitEncoder(nn.Module):
    def __init__(self, embedding_dim=32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, embedding_dim),
            nn.LayerNorm(embedding_dim),
        )

    def forward(self, images):
        return self.network(images)


def make_loader(images, labels=None, batch_size=256, shuffle=True, seed=1110):
    tensors = (images,) if labels is None else (images, labels)
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle,
        generator=generator,
    )


first_view = augment_batch(train_images[:16])
second_view = augment_batch(train_images[:16])
assert all_images.shape == (1797, 1, 8, 8)
assert len(set(train_idx) & set(test_idx)) == 0
assert first_view.shape == second_view.shape == (16, 1, 8, 8)
print({
    "split": (len(train_idx), len(val_idx), len(test_idx)),
    "pixel range": (float(all_images.min()), float(all_images.max())),
    "train class counts": torch.bincount(train_labels).tolist(),
})
```

</details>

Augmentation contract 被刻意设计得较温和，因为 8×8 数字的冗余很少。Contrast jitter、Gaussian noise 和稀疏 pixel dropout 通常保留身份，而 horizontal flip 可能把任务变成无效等价关系。因此，augmentation 本身就是学习目标的一部分：把两个 view 宣告为“相同”，实际上是在告诉 encoder 应删除哪些信息。


### **相似度、距离与度量学习** {#similarity-distance-metric-learning}

Embedding 通过比较规则变得可操作。Euclidean distance 测量绝对位移，

$$
d_2(h_i,h_j)=\lVert h_i-h_j\rVert_2,
$$

而 cosine similarity 测量夹角，

$$
s_{\cos}(h_i,h_j)=\frac{h_i^\top h_j}{\lVert h_i\rVert_2\lVert h_j\rVert_2}.
$$

Cosine similarity 忽略正比例缩放，适合方向编码语义、norm 反映置信度或频率的情况。Learned Mahalanobis metric $d_M^2=(h_i-h_j)^\top M(h_i-h_j)$ 可以在 $M\succeq0$ 时拉伸重要方向。Neural metric learning 通常把这种灵活性放进 encoder，再在学习空间使用 Euclidean 或 cosine distance。

Nearest-neighbor 行为由 representation、normalization、metric 和候选分布共同决定。高维距离可能集中；approximate nearest-neighbor index 用精确性换取延迟；在一个人群上校准的 similarity threshold 可能在 domain shift 下失效。因此，retrieval quality 必须使用与部署一致的 gallery composition 评估。

<details>
<summary><strong>Python：在真实 digit pixel 上比较 Euclidean 与 cosine neighbor</strong></summary>

```python
def l2_normalize(vectors):
    return vectors / vectors.norm(dim=1, keepdim=True).clamp_min(1e-8)


query = test_images[0].flatten().unsqueeze(0)
candidate_pixels = train_images.flatten(1)
euclidean_distance = torch.cdist(query, candidate_pixels).squeeze(0)
cosine_similarity = l2_normalize(query) @ l2_normalize(candidate_pixels).T

euclidean_neighbor = int(euclidean_distance.argmin())
cosine_neighbor = int(cosine_similarity.argmax())
scaled_query = 0.1 * query
scaled_euclidean = torch.cdist(scaled_query, candidate_pixels).squeeze(0)
scaled_cosine = l2_normalize(scaled_query) @ l2_normalize(candidate_pixels).T

assert torch.allclose(cosine_similarity, scaled_cosine, atol=1e-6)
assert euclidean_distance.shape == (len(train_images),)
print({
    "query label": int(test_labels[0]),
    "Euclidean neighbor label": int(train_labels[euclidean_neighbor]),
    "cosine neighbor label": int(train_labels[cosine_neighbor]),
    "Euclidean neighbor changes after scaling": bool(
        euclidean_neighbor != int(scaled_euclidean.argmin())
    ),
})
```

</details>

缩放 query 不会改变 cosine ranking，却可能改变 Euclidean ranking，因为 Euclidean distance 保留 magnitude。这不能证明 cosine 更好。如果 embedding norm 含有有效证据，归一化会删除信息；如果 norm 是不受控制的 artifact，归一化则提高稳定性。


### **Siamese Network 与 Triplet Loss** {#siamese-networks-triplet-loss}

Siamese network 对多个输入应用同一个共享 encoder。权重共享保证 embedding coordinate 对 anchor、positive 和 negative 表示相同含义。Pairwise contrastive loss 可以拉近匹配 pair，并把不匹配 pair 推到 margin 外。Triplet loss 直接约束相对顺序：

$$
\mathcal L_{\text{triplet}}
=\max\left(0,
d(h_a,h_p)-d(h_a,h_n)+m\right),
$$

其中 $a$ 是 anchor，$p$ 具有期望的相同语义，$n$ 应不同，margin $m>0$ 指定所需间隔。只有 negative 至少比 positive 远 $m$ 时，loss 才为零。[FaceNet](https://www.cv-foundation.org/openaccess/content_cvpr_2015/html/Schroff_FaceNet_A_Unified_2015_CVPR_paper.html)使 triplet training 在 verification 与 retrieval 中广为人知。

Triplet selection 主导优化。Easy triplet 已经具有零 loss，只会浪费计算。最难 negative 可能被错误标注或属于 outlier，导致训练不稳定。Semi-hard mining 选择比 positive 更远但仍处在 margin 内的 negative；batch-hard mining 在一个 batch 内选择高信息量样本。Mining 绝不能搜索 validation 或 test label。

<details>
<summary><strong>PyTorch：训练标签监督的 triplet embedding</strong></summary>

```python
def sample_triplets(images, labels, count, seed):
    generator = np.random.default_rng(seed)
    labels_np = labels.numpy()
    by_class = {label: np.flatnonzero(labels_np == label) for label in range(10)}
    anchors, positives, negatives = [], [], []
    for _ in range(count):
        anchor_index = int(generator.integers(len(images)))
        anchor_label = int(labels_np[anchor_index])
        positive_choices = by_class[anchor_label]
        positive_index = anchor_index
        while positive_index == anchor_index:
            positive_index = int(generator.choice(positive_choices))
        negative_label = int(generator.choice([x for x in range(10) if x != anchor_label]))
        negative_index = int(generator.choice(by_class[negative_label]))
        anchors.append(anchor_index)
        positives.append(positive_index)
        negatives.append(negative_index)
    return images[anchors], images[positives], images[negatives]


def triplet_satisfaction(encoder, triplets, margin=0.4):
    encoder.eval()
    with torch.no_grad():
        anchor, positive, negative = (l2_normalize(encoder(x)) for x in triplets)
        positive_distance = (anchor - positive).pow(2).sum(1)
        negative_distance = (anchor - negative).pow(2).sum(1)
    return float((positive_distance + margin < negative_distance).float().mean())


seed_everything(1112)
metric_encoder = DigitEncoder()
validation_triplets = sample_triplets(val_images, val_labels, 512, seed=1112)
before_satisfaction = triplet_satisfaction(metric_encoder, validation_triplets)
optimizer = torch.optim.AdamW(metric_encoder.parameters(), lr=2e-3, weight_decay=1e-4)
margin = 0.4
for epoch in range(35):
    metric_encoder.train()
    anchor, positive, negative = sample_triplets(
        train_images, train_labels, 1024, seed=1200 + epoch
    )
    anchor_embedding = l2_normalize(metric_encoder(anchor))
    positive_embedding = l2_normalize(metric_encoder(positive))
    negative_embedding = l2_normalize(metric_encoder(negative))
    loss = F.triplet_margin_loss(
        anchor_embedding, positive_embedding, negative_embedding, margin=margin
    )
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

after_satisfaction = triplet_satisfaction(metric_encoder, validation_triplets)
assert 0.0 <= after_satisfaction <= 1.0
print({"margin satisfaction before": round(before_satisfaction, 3),
       "after": round(after_satisfaction, 3),
       "final triplet loss": round(float(loss.detach()), 4)})
```

</details>

这个示例是**监督式度量学习**，不是自监督：positive 与 negative 身份来自 digit label。Validation statistic 报告固定 triplet 中满足严格 margin 的比例，比只看 training loss 更有诊断价值。它仍然不能测量 open-set calibration 或 writer population 变化后的 retrieval。


### **对比学习与 InfoNCE** {#contrastive-learning-infonce}

Contrastive SSL 用 transformation 或 paired observation 取代类别定义的 positive。同一样本的两个独立 augmented view 构成 positive pair；比较 dictionary 中的其他样本作为 negative。对 anchor $i$、positive $j$、归一化 projection $z$、temperature $\tau$ 和 candidate set $\mathcal A(i)$，InfoNCE 为

$$
\ell_{i,j}=-\log
\frac{\exp(z_i^\top z_j/\tau)}
{\sum_{k\in\mathcal A(i)}\exp(z_i^\top z_k/\tau)}.
$$

分子奖励两个 view 的**对齐（alignment）**。分母阻止全部样本坍缩到同一点，并推动表示分布展开；[Wang 与 Isola](https://arxiv.org/abs/2005.10242)从 alignment-uniformity 角度分析了这一点。较小的 $\tau$ 会使竞争更尖锐、放大 hard negative，也可能放大噪声与 false negative。

![Contrastive pipeline 把两个 augmentation 作为 positive pair，并把其他 batch sample 作为 negative。](assets/dl11-contrastive-pipeline.svg){fig-align="center" width="78%" fig-alt="一张源图像分成两个 augmentation，经过共享 encoder 得到两个 embedding，再进入 InfoNCE objective。"}

*原创教学图，依据 [SimCLR formulation](https://proceedings.mlr.press/v119/chen20j.html)以及 Wang 与 Isola 的 alignment-uniformity 分析。*

<details>
<summary><strong>PyTorch：实现 NT-Xent 并训练无标签 SimCLR encoder</strong></summary>

```python
class SimCLR(nn.Module):
    def __init__(self, embedding_dim=32, projection_dim=16):
        super().__init__()
        self.encoder = DigitEncoder(embedding_dim)
        self.projector = nn.Sequential(
            nn.Linear(embedding_dim, 64), nn.ReLU(), nn.Linear(64, projection_dim)
        )

    def forward(self, images):
        representation = self.encoder(images)
        projection = l2_normalize(self.projector(representation))
        return representation, projection


def nt_xent_loss(first_projection, second_projection, temperature=0.2):
    batch_size = len(first_projection)
    projections = torch.cat([first_projection, second_projection], dim=0)
    similarities = projections @ projections.T / temperature
    similarities.fill_diagonal_(-torch.inf)
    positive_target = torch.arange(2 * batch_size)
    positive_target = (positive_target + batch_size) % (2 * batch_size)
    return F.cross_entropy(similarities, positive_target)


seed_everything(1113)
simclr = SimCLR()
optimizer = torch.optim.AdamW(simclr.parameters(), lr=2e-3, weight_decay=1e-4)
epoch_losses = []
for epoch in range(45):
    simclr.train()
    losses = []
    for (images,) in make_loader(train_images, shuffle=True, seed=1300 + epoch):
        first = augment_batch(images)
        second = augment_batch(images)
        _, first_projection = simclr(first)
        _, second_projection = simclr(second)
        loss = nt_xent_loss(first_projection, second_projection, temperature=0.2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach()))
    epoch_losses.append(float(np.mean(losses)))

simclr_encoder = simclr.encoder.eval()
assert np.isfinite(epoch_losses).all()
print({"InfoNCE epoch 1": round(epoch_losses[0], 4),
       "epoch 45": round(epoch_losses[-1], 4)})
```

</details>

标签从未进入这个训练循环。然而，“label-free”不代表没有假设：augmentation code 声明 invariance，batch composition 定义 negative，projection head 决定 contrastive loss 作用的位置。下游任务保留 encoder representation，较小的 projection space 会被丢弃。


### **SimCLR 与 MoCo** {#simclr-moco}

[SimCLR](https://proceedings.mlr.press/v119/chen20j.html)使用共享 encoder、非线性 projection head、两个强 view 和 in-batch negative。Batch size 为 $B$ 时，对称 loss 比较 $2B$ 个 projection。它的简洁性伴随系统耦合：扩大 negative dictionary 需要更大的 batch 或跨设备通信。

[Momentum Contrast（MoCo）](https://arxiv.org/abs/1911.05722)把 dictionary size 与 batch size 解耦。Query encoder 接收 gradient；key encoder 通过 exponential moving average（EMA）跟随它，

$$
\theta_k\leftarrow \mu\theta_k+(1-\mu)\theta_q,
$$

FIFO queue 保存近期归一化 key 作为 negative。较大的 $\mu$ 使 key 缓慢变化，让 queue 中的 representation 保持相对一致。Stale queue 仍是一种权衡：它提高多样性，但包含旧 encoder 产生的 embedding。

![SimCLR、MoCo、BYOL 与 DINO 使用不同机制构造 target 并避免 collapse。](assets/dl11-ssl-families.svg){fig-align="center" width="78%" fig-alt="四个面板对比 SimCLR batch negative、MoCo momentum queue、BYOL prediction 和 DINO teacher distribution。"}

*原创对比图，依据 [SimCLR](https://proceedings.mlr.press/v119/chen20j.html)、[MoCo](https://arxiv.org/abs/1911.05722)、[BYOL](https://proceedings.neurips.cc/paper/2020/hash/f3ada80d5c4ee70142b17b8192b2958e-Abstract.html) 与 [DINO](https://arxiv.org/abs/2104.14294) 原始论文。*

<details>
<summary><strong>PyTorch：构造 momentum encoder 与循环 negative queue</strong></summary>

```python
seed_everything(1114)
query_encoder = copy.deepcopy(simclr_encoder)
key_encoder = copy.deepcopy(query_encoder)
for parameter in key_encoder.parameters():
    parameter.requires_grad_(False)

queue_size = 256
queue = l2_normalize(torch.randn(queue_size, 32))
queue_pointer = 0
query_optimizer = torch.optim.AdamW(query_encoder.parameters(), lr=8e-4)
momentum = 0.99

for step in range(24):
    indices = torch.randperm(len(train_images))[:64]
    query_view = augment_batch(train_images[indices])
    key_view = augment_batch(train_images[indices])
    queries = l2_normalize(query_encoder(query_view))
    with torch.no_grad():
        keys = l2_normalize(key_encoder(key_view))
    positive_logit = (queries * keys).sum(1, keepdim=True)
    negative_logits = queries @ queue.T
    logits = torch.cat([positive_logit, negative_logits], dim=1) / 0.2
    loss = F.cross_entropy(logits, torch.zeros(len(queries), dtype=torch.long))
    query_optimizer.zero_grad()
    loss.backward()
    query_optimizer.step()

    with torch.no_grad():
        for query_parameter, key_parameter in zip(
            query_encoder.parameters(), key_encoder.parameters()
        ):
            key_parameter.mul_(momentum).add_(query_parameter, alpha=1 - momentum)
        end = queue_pointer + len(keys)
        if end <= queue_size:
            queue[queue_pointer:end] = keys
        else:
            first_count = queue_size - queue_pointer
            queue[queue_pointer:] = keys[:first_count]
            queue[:end - queue_size] = keys[first_count:]
        queue_pointer = end % queue_size

assert queue.shape == (256, 32) and not queue.requires_grad
assert 0 <= queue_pointer < queue_size
print({"dictionary entries": queue_size, "batch queries": len(queries),
       "queue pointer": queue_pointer, "last MoCo loss": round(float(loss), 4)})
```

</details>

Queue 与 gradient 分离，key 由缓慢更新的 encoder 生成。把带 gradient 的 query embedding 加入 queue，或用普通 backpropagation 更新 key encoder，都会改变算法。在 distributed training 中，queue 和 batch statistics 还必须在 worker 之间保持一致。


### **非对比学习：BYOL 与 DINO** {#non-contrastive-learning-byol-dino}

Negative-free 方法追问：没有显式排斥其他样本，两个 view 是否仍能学习一致表示？[BYOL](https://proceedings.neurips.cc/paper/2020/hash/f3ada80d5c4ee70142b17b8192b2958e-Abstract.html)包含 online encoder-projector-predictor 与 EMA target encoder-projector。Online branch 预测另一 view 的 stop-gradient target representation，并交换两个方向。架构不对称、stop-gradient、normalization、optimization、augmentation 与 moving target 共同作用来避免 trivial collapse；“仅靠 EMA 防止 collapse”是过强结论。

[DINO](https://arxiv.org/abs/2104.14294)让 student probability distribution 匹配 EMA teacher 在不同 view 上的分布。Teacher logits 通过 centering 防止某个维度占据主导，并用较低 temperature sharpening 产生信息量更高的 target。Multi-crop augmentation 给 student 局部和全局 view，而 teacher 只看全局 view。DINO 的 emergent attention behavior 与其完整 Vision Transformer 设置有关，并非任何 teacher-student loss 都会自动产生。

<details>
<summary><strong>PyTorch：训练 BYOL regression 并构造 DINO-style target</strong></summary>

```python
class BYOLProjector(nn.Module):
    def __init__(self, input_dim=32, output_dim=32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Linear(64, output_dim),
        )

    def forward(self, features):
        return self.network(features)


def cosine_regression(prediction, target):
    prediction = l2_normalize(prediction)
    target = l2_normalize(target.detach())
    return 2 - 2 * (prediction * target).sum(1).mean()


seed_everything(1115)
online_encoder = DigitEncoder()
online_projector = BYOLProjector()
predictor = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 32))
target_encoder = copy.deepcopy(online_encoder)
target_projector = copy.deepcopy(online_projector)
for module in (target_encoder, target_projector):
    for parameter in module.parameters():
        parameter.requires_grad_(False)

optimizer = torch.optim.AdamW(
    list(online_encoder.parameters()) + list(online_projector.parameters())
    + list(predictor.parameters()), lr=1.5e-3, weight_decay=1e-4
)
byol_losses = []
for epoch in range(30):
    losses = []
    for (images,) in make_loader(train_images, shuffle=True, seed=1400 + epoch):
        first, second = augment_batch(images), augment_batch(images)
        first_prediction = predictor(online_projector(online_encoder(first)))
        second_prediction = predictor(online_projector(online_encoder(second)))
        with torch.no_grad():
            first_target = target_projector(target_encoder(first))
            second_target = target_projector(target_encoder(second))
        loss = 0.5 * (
            cosine_regression(first_prediction, second_target)
            + cosine_regression(second_prediction, first_target)
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            for online, target in zip(online_encoder.parameters(), target_encoder.parameters()):
                target.mul_(0.99).add_(online, alpha=0.01)
            for online, target in zip(online_projector.parameters(), target_projector.parameters()):
                target.mul_(0.99).add_(online, alpha=0.01)
        losses.append(float(loss.detach()))
    byol_losses.append(float(np.mean(losses)))

byol_encoder = online_encoder.eval()
# DINO uses centered, sharpened teacher distributions instead of vector regression.
student_head = nn.Linear(32, 10)
teacher_head = copy.deepcopy(student_head)
with torch.no_grad():
    teacher_logits = teacher_head(target_encoder(second_view))
    center = teacher_logits.mean(0, keepdim=True)
    teacher_probabilities = torch.softmax((teacher_logits - center) / 0.04, dim=1)
student_log_probabilities = torch.log_softmax(
    student_head(online_encoder(first_view)) / 0.1, dim=1
)
dino_style_loss = -(teacher_probabilities * student_log_probabilities).sum(1).mean()

assert torch.allclose(teacher_probabilities.sum(1), torch.ones(16), atol=1e-6)
print({"BYOL epoch 1 -> 30": (round(byol_losses[0], 4), round(byol_losses[-1], 4)),
       "DINO-style distribution loss": round(float(dino_style_loss), 4)})
```

</details>

这个小型 digits 实验实现机制，而不是复现 ImageNet 结果。BYOL loss 下降只说明 online prediction 能匹配 target feature；仍需 collapse diagnostic 与 downstream probe。DINO 代码演示 centering、sharpening 以及针对 stop-gradient distribution 的 cross-entropy，但不声称是完整 multi-crop ViT training run。


### **跨模态对齐与 CLIP** {#cross-modal-alignment-clip}

Cross-modal learning 从配对模态定义 positive。[CLIP](https://proceedings.mlr.press/v139/radford21a.html)使用独立 image encoder 与 text encoder，归一化两边输出，并构造 batch similarity matrix

$$
S_{ij}=\exp(s)\,\frac{v_i^\top t_j}{\lVert v_i\rVert\lVert t_j\rVert},
$$

其中学习到的 logit scale $s$ 作为 inverse temperature。对称 cross-entropy 要求每张 image 检索对应 text，同时每段 text 检索对应 image。推理时，natural-language prompt 可以成为 class prototype 或 retrieval query，把表示学习连接到 zero-shot transfer。

![Dual encoder 在同一个归一化 similarity space 中对齐 image 与 text。](assets/dl11-cross-modal.svg){fig-align="center" width="75%" fig-alt="Image tower 和 text tower 产生 embedding，在以匹配 pair 为对角线的 similarity matrix 中比较。"}

*原创教学图，依据 [Learning Transferable Visual Models From Natural Language Supervision](https://proceedings.mlr.press/v139/radford21a.html)。*

Digits dataset 只有 class label，没有自然 caption。因此代码把每个类别映射到 `digit seven` 等英文 prompt，并学习十个 text prototype。这是**标签监督的跨模态对齐**，不是忠实的 CLIP 预训练语料。它隔离 dual-encoder geometry，同时明确展示 supervision boundary。

<details>
<summary><strong>PyTorch：将 digit image 与十个 text prompt 对齐</strong></summary>

```python
digit_prompts = [
    "digit zero", "digit one", "digit two", "digit three", "digit four",
    "digit five", "digit six", "digit seven", "digit eight", "digit nine",
]


class TinyDualEncoder(nn.Module):
    def __init__(self, embedding_dim=32):
        super().__init__()
        self.image_encoder = DigitEncoder(embedding_dim)
        self.text_embeddings = nn.Embedding(len(digit_prompts), embedding_dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def forward(self, images):
        image_features = l2_normalize(self.image_encoder(images))
        prompt_ids = torch.arange(len(digit_prompts))
        text_features = l2_normalize(self.text_embeddings(prompt_ids))
        scale = self.logit_scale.exp().clamp(max=100)
        return scale * image_features @ text_features.T


seed_everything(1116)
dual_encoder = TinyDualEncoder()
optimizer = torch.optim.AdamW(dual_encoder.parameters(), lr=2e-3, weight_decay=1e-4)
for epoch in range(45):
    for images, labels in make_loader(
        train_images, train_labels, shuffle=True, seed=1500 + epoch
    ):
        logits = dual_encoder(augment_batch(images, noise_std=0.04, drop_probability=0.04))
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

dual_encoder.eval()
with torch.no_grad():
    validation_logits = dual_encoder(val_images)
    validation_accuracy = (validation_logits.argmax(1) == val_labels).float().mean()
clip_image_encoder = dual_encoder.image_encoder
assert validation_logits.shape == (len(val_images), 10)
print({"prompts": digit_prompts[:3] + ["..."],
       "validation prompt accuracy": round(float(validation_accuracy), 3),
       "learned temperature": round(float(dual_encoder.logit_scale.exp().reciprocal()), 4)})
```

</details>

真实 web-scale pair 含有噪声、重复、文化不均衡，而且并不相互独立。Batch negative 可能包含有效的替代 caption；prompt wording 会改变分数；zero-shot label 继承 paired data 中的偏差。Cross-modal scale 扩大覆盖范围，但不会把 noisy text 变成客观 semantic ground truth。


### **掩码表示学习** {#masked-representation-learning}

Masked learning 删除观察的一部分，并根据可见上下文预测它。对于 mask $M\in\{0,1\}^D$，reconstruction objective 可以只评估隐藏坐标，

$$
\mathcal L_{\text{mask}}=
\frac{1}{\sum_d M_d}\sum_{d=1}^{D}M_d\left(x_d-\hat x_d\right)^2.
$$

Language model 预测 masked token 或 next token；image model 预测 pixel、discrete visual token 或 latent target。[Masked Autoencoders（MAE）](https://arxiv.org/abs/2111.06377)采用非对称设计：较重 encoder 只处理 visible patch，较轻 decoder 重建 missing pixel。图像包含大量冗余时，高 masking ratio 既能降低 encoder cost，也能避免任务退化为局部复制。

![Masked encoder 必须概括可见证据，使 decoder 能重建隐藏坐标。](assets/dl11-masked-learning.svg){fig-align="center" width="76%" fig-alt="原图、binary mask、encoder latent、decoder reconstruction，以及只在 masked position 上计算的 loss。"}

*原创教学图，依据 [Masked Autoencoders](https://arxiv.org/abs/2111.06377) objective。*

<details>
<summary><strong>PyTorch：无标签训练 masked digit autoencoder</strong></summary>

```python
class MaskedDigitAutoencoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(64, 96), nn.ReLU(), nn.Linear(96, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 96), nn.ReLU(), nn.Linear(96, 64), nn.Sigmoid()
        )

    def forward(self, flattened, mask):
        masked_input = torch.where(mask, torch.full_like(flattened, -1.0), flattened)
        latent = self.encoder(masked_input)
        return self.decoder(latent), latent


seed_everything(1117)
masked_autoencoder = MaskedDigitAutoencoder()
optimizer = torch.optim.AdamW(masked_autoencoder.parameters(), lr=2e-3)
mask_ratio = 0.45
for epoch in range(55):
    for (images,) in make_loader(train_images, shuffle=True, seed=1600 + epoch):
        flattened = images.flatten(1)
        mask = torch.rand_like(flattened) < mask_ratio
        reconstruction, _ = masked_autoencoder(flattened, mask)
        loss = F.mse_loss(reconstruction[mask], flattened[mask])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

test_flattened = test_images.flatten(1)
generator = torch.Generator().manual_seed(1117)
test_mask = torch.rand(test_flattened.shape, generator=generator) < mask_ratio
masked_autoencoder.eval()
with torch.no_grad():
    test_reconstruction, masked_embeddings = masked_autoencoder(test_flattened, test_mask)
    reconstruction_mse = F.mse_loss(
        test_reconstruction[test_mask], test_flattened[test_mask]
    )
    train_pixel_mean = train_images.flatten(1).mean(0)
    baseline_mse = F.mse_loss(
        train_pixel_mean.expand_as(test_flattened)[test_mask], test_flattened[test_mask]
    )

assert masked_embeddings.shape == (len(test_images), 32)
print({"masked fraction": round(float(test_mask.float().mean()), 3),
       "model masked MSE": round(float(reconstruction_mse), 4),
       "train-mean baseline MSE": round(float(baseline_mse), 4)})
```

</details>

`-1` mask sentinel 位于有效 pixel range 之外，因此 encoder 能区分隐藏的零与可见的背景零。模型与 train-set pixel-mean reconstruction 比较，避免只凭较低 MSE 就误判模型有意义。Reconstruction quality 与 representation utility 并不相同：decoder 可能奖励 classifier 应忽略的纹理细节。


### **知识蒸馏** {#knowledge-distillation}

Knowledge distillation 训练较小 student 复现 teacher 的 predictive distribution 或 intermediate representation。对 class logits $z_t,z_s$、temperature $T$、hard label $y$ 和 mixing coefficient $\alpha$，

$$
\mathcal L=(1-\alpha)\operatorname{CE}(y,z_s)
+\alpha T^2\operatorname{KL}\!\left(
\operatorname{softmax}(z_t/T)\;\Vert\;
\operatorname{softmax}(z_s/T)
\right).
$$

Temperature 揭示 non-target class 之间的相对概率，有时称为“dark knowledge”。$T^2$ 因子补偿 softened probability 导致的 gradient shrinkage。[Hinton、Vinyals 与 Dean](https://arxiv.org/abs/1503.02531)为压缩 ensemble 与大型网络发展了这一 formulation。

Distillation 不限于 final logits。Feature distillation 对齐 hidden state，attention transfer 对齐 map，self-distillation 使用同一架构的另一个 checkpoint 或 EMA branch。不能期待 student 超越 teacher 和 data 中不存在的信息；它还可能继承 teacher 的 calibration error 与 bias。

<details>
<summary><strong>PyTorch：把 digit prompt teacher 蒸馏到更小的 student</strong></summary>

```python
for parameter in dual_encoder.parameters():
    parameter.requires_grad_(False)


class SmallStudent(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 24), nn.ReLU(), nn.Linear(24, 10)
        )

    def forward(self, images):
        return self.network(images)


def train_student(use_distillation, seed):
    seed_everything(seed)
    student = SmallStudent()
    optimizer = torch.optim.AdamW(student.parameters(), lr=2e-3)
    temperature, soft_weight = 3.0, 0.7
    for epoch in range(45):
        for images, labels in make_loader(
            train_images, train_labels, shuffle=True, seed=1700 + epoch
        ):
            student_logits = student(images)
            hard_loss = F.cross_entropy(student_logits, labels)
            if use_distillation:
                with torch.no_grad():
                    teacher_logits = dual_encoder(images)
                soft_loss = F.kl_div(
                    F.log_softmax(student_logits / temperature, dim=1),
                    F.softmax(teacher_logits / temperature, dim=1),
                    reduction="batchmean",
                ) * temperature**2
                loss = (1 - soft_weight) * hard_loss + soft_weight * soft_loss
            else:
                loss = hard_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    student.eval()
    with torch.no_grad():
        accuracy = (student(test_images).argmax(1) == test_labels).float().mean()
    return student, float(accuracy)


hard_student, hard_accuracy = train_student(False, 1118)
distilled_student, distilled_accuracy = train_student(True, 1118)
teacher_parameters = sum(parameter.numel() for parameter in dual_encoder.parameters())
student_parameters = sum(parameter.numel() for parameter in distilled_student.parameters())

assert student_parameters < teacher_parameters
print({"teacher/student parameters": (teacher_parameters, student_parameters),
       "hard-label accuracy": round(hard_accuracy, 3),
       "distilled accuracy": round(distilled_accuracy, 3)})
```

</details>

Hard-only 与 distilled student 从相同 seed 开始，并使用同一 split。一次小型运行不一定出现 distillation gain，因此代码只报告而不断言优势。有效结论是：soft teacher probability 可以与 hard label 结合，同时降低 parameter count；latency 与 memory 仍应在部署硬件上测量。


### **Embedding Retrieval 与表示评估** {#embedding-retrieval-representation-evaluation}

SSL loss 是 pretraining diagnostic，不是 representation quality 的最终测量。评估应冻结 encoder，并检查简单 downstream mechanism 能否使用其输出。**Linear probe**只训练一个线性分类器，测试标签是否近似线性可访问。$k$-nearest-neighbor classifier 不拟合深层 head，直接测试局部几何。Retrieval 根据相关候选是一个还是多个，报告 Recall@$K$、precision、mAP 或 NDCG。

协议细节很重要。Probe hyperparameter 必须在 validation data 上选择；test label 只访问一次。所有 representation 使用相同 split 与 preprocessing。强大的 nonlinear probe 可能通过重新学习任务掩盖弱表示，而过小 linear probe 也可能 underfit。Fine-tuning 衡量 adaptation performance，而不是 frozen representation quality，属于下一章。

<details>
<summary><strong>Python：运行 validation-selected linear probe、5-NN 与 retrieval</strong></summary>

```python
def encode_in_batches(encoder, images, batch_size=256):
    encoder.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            outputs.append(encoder(images[start:start + batch_size]))
    return torch.cat(outputs)


train_embeddings = l2_normalize(encode_in_batches(simclr_encoder, train_images)).numpy()
val_embeddings = l2_normalize(encode_in_batches(simclr_encoder, val_images)).numpy()
test_embeddings = l2_normalize(encode_in_batches(simclr_encoder, test_images)).numpy()

# Select the linear-probe regularization on validation data, then touch test once.
best_validation_accuracy, best_probe = -1.0, None
for regularization in (0.1, 1.0, 10.0):
    probe = LogisticRegression(C=regularization, max_iter=1000, random_state=1119)
    probe.fit(train_embeddings, train_labels.numpy())
    score = probe.score(val_embeddings, val_labels.numpy())
    if score > best_validation_accuracy:
        best_validation_accuracy, best_probe = score, probe
linear_probe_accuracy = best_probe.score(test_embeddings, test_labels.numpy())

knn = KNeighborsClassifier(n_neighbors=5, metric="cosine", weights="distance")
knn.fit(train_embeddings, train_labels.numpy())
knn_accuracy = knn.score(test_embeddings, test_labels.numpy())

similarities = test_embeddings @ train_embeddings.T
ranking = np.argsort(-similarities, axis=1)
recall_at_1 = np.mean(
    train_labels.numpy()[ranking[:, :1]] == test_labels.numpy()[:, None]
)
recall_at_5 = np.mean(
    np.any(train_labels.numpy()[ranking[:, :5]] == test_labels.numpy()[:, None], axis=1)
)

assert 0 <= linear_probe_accuracy <= 1 and recall_at_5 >= recall_at_1
print({"linear probe validation/test": (round(best_validation_accuracy, 3), round(linear_probe_accuracy, 3)),
       "5-NN test accuracy": round(knn_accuracy, 3),
       "retrieval Recall@1/5": (round(float(recall_at_1), 3), round(float(recall_at_5), 3))})
```

</details>

这里的 Recall@1 询问最近训练图像是否具有相同 digit class；Recall@5 询问五个最近候选中是否至少有一个同类。这种 class-based relevance definition 适合教学数据集，却不同于 instance retrieval；在后者中，同类别的另一张图也可能算错误。选择指标前必须先定义 relevance。


### **失败模式与捷径学习** {#failure-modes-shortcut-learning}

Representation learning 可能优化 pretext objective，却学到错误 invariance。如果 positive view 共享边框、watermark、采集设备或 preprocessing artifact，encoder 可能匹配捷径而不是 semantic content。Augmentation 太弱时，instance identity 过于简单；太强时，positive 不再保留目标概念。定义有效 transformation 需要 domain knowledge。

Contrastive method 在语义相关样本被当成不同 instance 时会遇到 **false negative**。Dictionary 太小会导致竞争不足；极难 negative 可能被错误标注。Non-contrastive method 面临**表示坍缩（representation collapse）**，即所有输入映射到一个 vector；还面临**维度坍缩（dimensional collapse）**，即方差只存在于少数方向。Feature standard deviation、covariance spectrum、effective rank、pairwise cosine similarity 与 downstream probe 分别揭示 collapse 的不同侧面。

其他 shortcut 来自 evaluation：在全部数据上拟合 normalization、根据 test retrieval 选择 checkpoint、把近重复 writer 放入不同 split，或使用 label 设计“self-supervised” pair 却不披露。Metric code 应附带 provenance assertion。

<details>
<summary><strong>PyTorch：诊断 collapse、alignment 与 false negative</strong></summary>

```python
def representation_diagnostics(embeddings):
    centered = embeddings - embeddings.mean(0, keepdim=True)
    covariance = centered.T @ centered / max(len(embeddings) - 1, 1)
    eigenvalues = torch.linalg.eigvalsh(covariance).clamp_min(0)
    probabilities = eigenvalues / eigenvalues.sum().clamp_min(1e-12)
    effective_rank = torch.exp(
        -(probabilities * probabilities.clamp_min(1e-12).log()).sum()
    )
    normalized = l2_normalize(embeddings)
    off_diagonal_similarity = (
        (normalized @ normalized.T).sum() - len(embeddings)
    ) / (len(embeddings) * (len(embeddings) - 1))
    return {
        "mean feature std": float(embeddings.std(0).mean()),
        "effective rank": float(effective_rank),
        "mean off-diagonal cosine": float(off_diagonal_similarity),
    }


with torch.no_grad():
    simclr_test = simclr_encoder(test_images[:128])
    byol_test = byol_encoder(test_images[:128])
    collapsed = torch.ones_like(simclr_test)
    view_one = l2_normalize(simclr_encoder(augment_batch(test_images[:128])))
    view_two = l2_normalize(simclr_encoder(augment_batch(test_images[:128])))
    positive_alignment = (view_one * view_two).sum(1).mean()

batch_labels = test_labels[:128]
same_semantic_class = batch_labels[:, None] == batch_labels[None, :]
false_negative_rate = float(
    (same_semantic_class & ~torch.eye(len(batch_labels), dtype=torch.bool)).float().mean()
)

diagnostics = {
    "SimCLR": representation_diagnostics(simclr_test),
    "BYOL": representation_diagnostics(byol_test),
    "collapsed control": representation_diagnostics(collapsed),
}
assert diagnostics["collapsed control"]["effective rank"] <= 1.01
print({name: {key: round(value, 3) for key, value in report.items()}
       for name, report in diagnostics.items()})
print({"positive-view cosine": round(float(positive_alignment), 3),
       "same-class pairs treated as negatives": round(false_negative_rate, 3)})
```

</details>

All-ones control 的 effective rank 接近 1，展示完整 collapse 的样子。健康的 effective rank 本身仍不充分：随机噪声可以占满所有维度，却不编码有效语义。反过来，某个任务也可能确实只需低维表示。Collapse diagnostic、pretext loss、linear probe、retrieval、robustness 与 transfer evidence 必须结合解释。


### **章节对比与总结** {#chapter-comparison-summary}

表示学习是通过几何表达的监督设计。Positive pair 决定哪些因素应变得 invariant；negative 或 anti-collapse mechanism 决定哪些内容必须保持不同；architecture 与 projection head 决定信息在哪里流动；evaluation 决定这种几何是否在 pretext task 之外有用。

| 方法 | 训练信号 | Collapse/竞争机制 | 主要优势 | 主要风险 |
|---|---|---|---|---|
| Siamese/triplet | 有标签的 positive 与 negative identity | 对选定 negative 施加显式 margin | 直接优化 verification/retrieval ordering | Mining bias、错误 hard negative 与标签成本 |
| SimCLR/InfoNCE | 每个 instance 的两个 augmented view | In-batch negative 与归一化 temperature-scaled logits | 目标简单、强大且透明 | Batch-size coupling 与 false negative |
| MoCo | Positive view 加 queued key | 大型 FIFO dictionary 与 momentum key encoder | 使用适中 batch 获得大量 negative | Stale key 与 distributed queue consistency |
| BYOL | Online prediction 匹配 EMA target view | Stop-gradient、predictor、EMA、normalization 与 augmentation | 不需要显式 negative | 必须诊断 collapse；多种机制相互作用 |
| DINO | Student distribution 匹配 centered/sharpened EMA teacher | Centering、sharpening、EMA 与 multi-crop design | 强大的 self-distilled semantic feature | 对 schedule 敏感；emergent behavior 依赖完整设置 |
| CLIP-style alignment | 配对 image 与 text observation | Cross-modal batch competition | 跨模态 retrieval 与 prompt-based transfer | Pair noise、prompt sensitivity、文化与 exposure bias |
| Masked modeling | 重建隐藏 token、pixel 或 latent | Masking 构造的信息 bottleneck | 可扩展使用无标签上下文 | Reconstruction detail 不等于 semantic utility |
| Knowledge distillation | Teacher logits 或 hidden feature | Teacher distribution 约束 student | 压缩与知识迁移 | Student 继承 teacher error 与 bias |

共享 digits 实验维护了明确的 supervision ledger。Triplet sampling 使用 label，并被称为监督式方法。SimCLR、MoCo、BYOL 与 masked reconstruction 只使用训练图像。Prompt alignment 使用 digit name，并被称为标签监督。Linear probe 与 retrieval 只在 encoder 冻结后使用 label。这样的 ledger 比把所有方法简单称为“unsupervised”更有信息量。

实际工作流是：

1. 在选择 augmentation 前定义 downstream task 与有效 invariance；
2. 建立 raw-feature、random-encoder 和 supervised baseline；
3. 根据可用结构选择 contrastive、teacher-student、masked 或 cross-modal target；
4. 同时监控 pretext loss、variance、effective rank、alignment 与 uniformity；
5. 冻结 encoder，在固定 split 上评估 linear probe、$k$-NN、retrieval、calibration 与 robustness；
6. 审计 label、pair、negative、augmentation、duplicate entity 与 test access；
7. 然后再决定是否有必要进行 full fine-tuning 或 parameter-efficient adaptation。

下一章以此为基础：表示完成预训练与评估后，transfer learning 将决定为了新 domain 和 task，应冻结、更新或适配其中多少部分。
